# `indic/03` — Compute Neural MT Metrics (COMET, BLEURT, BERTScore)

**Purpose:** Score each MT hypothesis in the IndicMT Eval corpus against its
human reference using three neural metrics — COMET, BLEURT, and BERTScore —
under two conditions: native script and romanised script.

**Reads:** `data/indic/<ISO>_indicmt.csv` (produced by `indic/01` and `indic/02`)

**Outputs produced:**
```
data/indic/GUJ_indicmt.csv   -- adds columns: comet, bleurt, bertscore_f1,
data/indic/HIN_indicmt.csv      bertscore_p, bertscore_r (native + _rom variants)
data/indic/MAL_indicmt.csv
data/indic/MAR_indicmt.csv
data/indic/TAM_indicmt.csv
data/mateo/<Language>/       -- source/reference/translation .csv + .tsv files
results/<ISO>_metrics.csv    -- per-language checkpoint CSVs
```

**Two scoring paths:**
- **MATEO (recommended for first use):** Upload per-language files to
  https://mateo.ivdnt.org/Evaluate and download results. The export section
  (Step 3) generates files in exactly the format MATEO requires.
- **Local scoring:** Steps 4–6 compute all three metrics directly on CPU or
  GPU using the same model checkpoints.

**Models used:**

| Metric | Model | Version |
|--------|-------|---------|
| COMET | `Unbabel/wmt22-comet-da` | unbabel-comet 2.2.6 |
| BLEURT | `BLEURT-20` | bleurt @ cebe7e6 |
| BERTScore | `microsoft/mdeberta-v3-base` | bert-score 0.3.12 |

---

**References**

- COMET: Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural
  Framework for MT Evaluation. *EMNLP 2020*, pp. 2685–2702.
  https://aclanthology.org/2020.emnlp-main.213

- BLEURT: Sellam, T., Das, D., & Parikh, A. (2020). BLEURT: Learning Robust Metrics
  for Text Generation. *ACL 2020*, pp. 7881–7892.
  https://aclanthology.org/2020.acl-main.704

- BERTScore: Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., & Artzi, Y. (2020).
  BERTScore: Evaluating Text Generation with BERT. *ICLR 2020*.
  https://arxiv.org/abs/1904.09675

- IndicMT Eval: Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P.,
  Khapra, M. M., & Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate
  Machine Translation Metrics for Indian Languages. *ACL 2023*, pp. 14210–14228.
  https://aclanthology.org/2023.acl-long.795

- MATEO: Vanroy, B., Tezcan, A., & Macken, L. (2023). MATEO: MAchine Translation
  Evaluation Online. *EAMT 2023*, pp. 499–500.
  https://aclanthology.org/2023.eamt-1.52

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}..."); _install(pkg)

# Neural metric dependencies — install once, then comment out
# Run in this exact order to avoid version conflicts:
#
# !pip install "transformers==4.40.2"
# !pip install "protobuf==4.25.3"
# !pip install "bert-score==0.3.12" --force-reinstall --no-deps
# !pip install git+https://github.com/google-research/bleurt.git@cebe7e6
# !pip install "unbabel-comet==2.2.6"

print("Dependencies ready.")

## Configuration

All paths are relative to the repository root.  
Set `DEVICE` to `"cuda"` if a GPU is available — COMET and BLEURT benefit
significantly from GPU acceleration at 7,000 sentences.

Set `SCORE_NATIVE` and `SCORE_ROM` to control which conditions are scored.
Both are `True` by default to reproduce the native-vs-romanised comparison.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ── protobuf fix: must be set before importing TensorFlow / BLEURT ─────────
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR    = Path("data/indic")
MATEO_DIR   = Path("data/mateo")
RESULTS_DIR = Path("results")
for d in [MATEO_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Language config: ISO code -> human-readable name ───────────────────────
LANG_CONFIGS = {
    "GUJ": "Gujarati",
    "HIN": "Hindi",
    "MAL": "Malayalam",
    "MAR": "Marathi",
    "TAM": "Tamil",
}

# ── Scoring conditions ─────────────────────────────────────────────────────
SCORE_NATIVE = True   # score native-script hyp/ref
SCORE_ROM    = True   # score romanised hyp_rom/ref_rom

# ── Device ─────────────────────────────────────────────────────────────────
# Change to "cuda" if a GPU is available
DEVICE = "cpu"  # Options: "cpu" | "cuda"

# ── Model identifiers ──────────────────────────────────────────────────────
COMET_MODEL      = "Unbabel/wmt22-comet-da"
BLEURT_CKPT      = "BLEURT-20"
BERTSCORE_MODEL  = "microsoft/mdeberta-v3-base"

print(f"Data directory  : {DATA_DIR.resolve()}")
print(f"MATEO directory : {MATEO_DIR.resolve()}")
print(f"Results dir     : {RESULTS_DIR.resolve()}")
print(f"Device          : {DEVICE}")
print(f"Score native    : {SCORE_NATIVE}")
print(f"Score romanised : {SCORE_ROM}")

## Step 1 — Load Per-Language CSVs

Read the five CSVs written by `indic/01` (with romanised columns added by
`indic/02`). Confirm that `hyp`, `ref`, `hyp_rom`, and `ref_rom` are all
present before scoring begins.

In [ ]:
data = {}  # iso -> pd.DataFrame

REQUIRED_NATIVE = ["src", "hyp", "ref"]
REQUIRED_ROM    = ["hyp_rom", "ref_rom"]

for iso in LANG_CONFIGS:
    path = DATA_DIR / f"{iso}_indicmt.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run indic/01 and indic/02 first.")
    df = pd.read_csv(path)
    data[iso] = df

    native_ok = all(c in df.columns for c in REQUIRED_NATIVE)
    rom_ok    = all(c in df.columns for c in REQUIRED_ROM)
    print(f"  {iso}  {len(df):,} rows  |  "
          f"native cols: {'OK' if native_ok else 'MISSING'}  |  "
          f"rom cols: {'OK' if rom_ok else 'MISSING (run indic/02)'}")

print(f"\nLoaded {len(data)} language files.")

## Step 2 — Export Files for MATEO

[MATEO](https://mateo.ivdnt.org/) is a web tool that computes BERTScore,
BLEURT, and COMET without local installation. It expects three plain-text
files per language (source / reference / translation), one sentence per
line, no header, UTF-8 encoded.

This step exports those files for both the **native** and **romanised**
conditions into `data/mateo/<ISO>/native/` and `data/mateo/<ISO>/romanised/`.

**How to use MATEO:**
1. Go to https://mateo.ivdnt.org/Evaluate
2. Upload `source.txt`, `reference.txt`, and `translation.txt` for one
   language at a time
3. Select metrics: BERTScore, BLEURT, COMET
4. Click **Evaluate**, download the results CSV
5. Repeat for each language and each condition (native / romanised)
6. Feed the downloaded scores into Step 3 (merge) below

**Citing MATEO:**
```
Vanroy, B., Tezcan, A., & Macken, L. (2023). MATEO: MAchine Translation
Evaluation Online. EAMT 2023, pp. 499–500.
https://aclanthology.org/2023.eamt-1.52
```

In [ ]:
def export_mateo(df: pd.DataFrame, out_dir: Path,
                 src_col: str, hyp_col: str, ref_col: str) -> None:
    """
    Write source / translation / reference as plain-text files for MATEO.
    Each file: one sentence per line, no header, UTF-8.
    Both .txt and .tsv versions are written.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    for fname, col in [("source", src_col),
                       ("translation", hyp_col),
                       ("reference", ref_col)]:
        series = df[col].fillna("").astype(str)
        series.to_csv(out_dir / f"{fname}.txt",
                      index=False, header=False, encoding="utf-8")
        series.to_csv(out_dir / f"{fname}.tsv",
                      index=False, header=False, sep="\t", encoding="utf-8")


for iso, df in data.items():
    lang_name = LANG_CONFIGS[iso]

    if SCORE_NATIVE and all(c in df.columns for c in ["src", "hyp", "ref"]):
        out = MATEO_DIR / iso / "native"
        export_mateo(df, out, "src", "hyp", "ref")
        print(f"  {iso} native    -> {out}/")

    if SCORE_ROM and all(c in df.columns for c in ["src", "hyp_rom", "ref_rom"]):
        out = MATEO_DIR / iso / "romanised"
        export_mateo(df, out, "src", "hyp_rom", "ref_rom")
        print(f"  {iso} romanised -> {out}/")

print(f"\nMATEO files ready in: {MATEO_DIR.resolve()}")
print("Upload each subfolder separately to https://mateo.ivdnt.org/Evaluate")

## Step 3 — Merge Pre-computed MATEO Scores (Optional)

If you ran scoring via MATEO, download the results CSVs and place them at:
```
data/mateo/<ISO>/native/mateo_results.csv
data/mateo/<ISO>/romanised/mateo_results.csv
```

Run this cell to merge the scores into the per-language CSVs. Skip this
step if you are computing metrics locally (Steps 4–6).

Expected MATEO output columns: `COMET`, `BLEURT`, `BERTScore`
(column names vary by MATEO version — adjust `COL_MAP` below if needed).

In [ ]:
# Column name mapping from MATEO output to our canonical names
COL_MAP = {
    "COMET":      "comet",
    "BLEURT":     "bleurt",
    "BERTScore":  "bertscore_f1",
}

for iso in LANG_CONFIGS:
    df = data[iso].copy()

    for condition in ["native", "romanised"]:
        results_path = MATEO_DIR / iso / condition / "mateo_results.csv"
        if not results_path.exists():
            print(f"  [{iso} {condition}] mateo_results.csv not found -- skipped")
            continue

        scores = pd.read_csv(results_path)
        suffix = "" if condition == "native" else "_rom"

        for mateo_col, canon_col in COL_MAP.items():
            if mateo_col in scores.columns:
                df[f"{canon_col}{suffix}"] = scores[mateo_col].values

        print(f"  [{iso} {condition}] merged {list(COL_MAP.values())} (suffix='{suffix}')")

    data[iso] = df

print("\nMATEO merge complete (skipped languages had no results file).")

## Step 4 — Local Scoring: COMET

Compute COMET scores locally using `Unbabel/wmt22-comet-da` (standard DA
model, unbabel-comet 2.2.6). COMET requires source + hypothesis + reference
for each sentence.

Scores are computed for all five languages in sequence. An intermediate
checkpoint CSV is saved after each language so progress is not lost if
a later step fails.

**CPU runtime:** ~20–40 min for 7,000 × 2 conditions.  
**GPU (CUDA):** ~3–6 min. Set `DEVICE = "cuda"` in Configuration.

In [ ]:
from comet import download_model, load_from_checkpoint

gpus = 0 if DEVICE == "cpu" else 1

print(f"Downloading / loading COMET model: {COMET_MODEL}")
comet_path  = download_model(COMET_MODEL)
comet_model = load_from_checkpoint(comet_path)
print("COMET model ready.\n")


def score_comet(df: pd.DataFrame, src_col: str,
                hyp_col: str, ref_col: str) -> list[float]:
    """Return a list of COMET segment scores (0–100 scale)."""
    records = [
        {"src": str(s), "mt": str(h), "ref": str(r)}
        for s, h, r in zip(df[src_col], df[hyp_col], df[ref_col])
    ]
    output = comet_model.predict(records, batch_size=64, gpus=gpus)
    # wmt22-comet-da returns scores in [0, 1]; multiply by 100 to match
    # the scale reported in the paper (Table 2)
    return [round(s * 100, 4) for s in output.scores]


for iso, df in data.items():
    print(f"{'='*55}")
    print(f"{iso} — COMET")
    print(f"{'='*55}")

    if SCORE_NATIVE and all(c in df.columns for c in ["src", "hyp", "ref"]):
        if "comet" not in df.columns:
            df["comet"] = score_comet(df, "src", "hyp", "ref")
            print(f"  native   mean COMET = {df['comet'].mean():.2f}")
        else:
            print(f"  native   'comet' already present -- skipped")

    if SCORE_ROM and all(c in df.columns for c in ["src", "hyp_rom", "ref_rom"]):
        if "comet_rom" not in df.columns:
            df["comet_rom"] = score_comet(df, "src", "hyp_rom", "ref_rom")
            print(f"  romanised mean COMET = {df['comet_rom'].mean():.2f}")
        else:
            print(f"  romanised 'comet_rom' already present -- skipped")

    data[iso] = df

    # Checkpoint save
    ckpt = RESULTS_DIR / f"{iso}_metrics.csv"
    df.to_csv(ckpt, index=False)
    print(f"  [checkpoint] {ckpt}")
    print()

## Step 5 — Local Scoring: BLEURT

Compute BLEURT scores using the `BLEURT-20` checkpoint (~1.2 GB).
The model is downloaded automatically on first run. Manual download:
https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip

**Reference:** Sellam, T., Das, D., & Parikh, A. (2020). BLEURT: Learning
Robust Metrics for Text Generation. *ACL 2020*, pp. 7881–7892.
https://aclanthology.org/2020.acl-main.704

In [ ]:
from bleurt import score as bleurt_score

print(f"Loading BLEURT checkpoint: {BLEURT_CKPT}")
bleurt_scorer = bleurt_score.BleurtScorer(BLEURT_CKPT)
print("BLEURT scorer ready.\n")


def score_bleurt(df: pd.DataFrame, hyp_col: str, ref_col: str) -> list[float]:
    """Return a list of BLEURT scores."""
    hyps = df[hyp_col].fillna("").astype(str).tolist()
    refs = df[ref_col].fillna("").astype(str).tolist()
    scores = bleurt_scorer.score(references=refs, candidates=hyps, batch_size=64)
    return [round(s, 4) for s in scores]


for iso, df in data.items():
    print(f"{'='*55}")
    print(f"{iso} — BLEURT")
    print(f"{'='*55}")

    if SCORE_NATIVE and all(c in df.columns for c in ["hyp", "ref"]):
        if "bleurt" not in df.columns:
            df["bleurt"] = score_bleurt(df, "hyp", "ref")
            print(f"  native    mean BLEURT = {df['bleurt'].mean():.4f}")
        else:
            print(f"  native    'bleurt' already present -- skipped")

    if SCORE_ROM and all(c in df.columns for c in ["hyp_rom", "ref_rom"]):
        if "bleurt_rom" not in df.columns:
            df["bleurt_rom"] = score_bleurt(df, "hyp_rom", "ref_rom")
            print(f"  romanised mean BLEURT = {df['bleurt_rom'].mean():.4f}")
        else:
            print(f"  romanised 'bleurt_rom' already present -- skipped")

    data[iso] = df

    ckpt = RESULTS_DIR / f"{iso}_metrics.csv"
    df.to_csv(ckpt, index=False)
    print(f"  [checkpoint] {ckpt}")
    print()

## Step 6 — Local Scoring: BERTScore

Compute BERTScore (P / R / F1) using `microsoft/mdeberta-v3-base` (~900 MB).
`model_type` is specified explicitly to bypass an AutoModel resolution issue
in `transformers >= 4.41`.

**Reference:** Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., & Artzi, Y.
(2020). BERTScore: Evaluating Text Generation with BERT. *ICLR 2020*.
https://arxiv.org/abs/1904.09675

In [ ]:
from bert_score import score as bertscore_fn

print(f"BERTScore model : {BERTSCORE_MODEL}")
print(f"Device          : {DEVICE}\n")


def score_bertscore(df: pd.DataFrame,
                    hyp_col: str, ref_col: str) -> tuple[list, list, list]:
    """Return (P, R, F1) lists of BERTScore values."""
    hyps = df[hyp_col].fillna("").astype(str).tolist()
    refs = df[ref_col].fillna("").astype(str).tolist()
    P, R, F1 = bertscore_fn(
        cands=hyps,
        refs=refs,
        model_type=BERTSCORE_MODEL,
        verbose=True,
        batch_size=64,
        device=DEVICE,
    )
    return (
        [round(v, 4) for v in P.tolist()],
        [round(v, 4) for v in R.tolist()],
        [round(v, 4) for v in F1.tolist()],
    )


for iso, df in data.items():
    print(f"{'='*55}")
    print(f"{iso} — BERTScore")
    print(f"{'='*55}")

    if SCORE_NATIVE and all(c in df.columns for c in ["hyp", "ref"]):
        if "bertscore_f1" not in df.columns:
            P, R, F1 = score_bertscore(df, "hyp", "ref")
            df["bertscore_p"]  = P
            df["bertscore_r"]  = R
            df["bertscore_f1"] = F1
            print(f"  native    mean F1 = {df['bertscore_f1'].mean():.4f}")
        else:
            print(f"  native    'bertscore_f1' already present -- skipped")

    if SCORE_ROM and all(c in df.columns for c in ["hyp_rom", "ref_rom"]):
        if "bertscore_f1_rom" not in df.columns:
            P, R, F1 = score_bertscore(df, "hyp_rom", "ref_rom")
            df["bertscore_p_rom"]  = P
            df["bertscore_r_rom"]  = R
            df["bertscore_f1_rom"] = F1
            print(f"  romanised mean F1 = {df['bertscore_f1_rom'].mean():.4f}")
        else:
            print(f"  romanised 'bertscore_f1_rom' already present -- skipped")

    data[iso] = df

    ckpt = RESULTS_DIR / f"{iso}_metrics.csv"
    df.to_csv(ckpt, index=False)
    print(f"  [checkpoint] {ckpt}")
    print()

## Step 7 — Save Updated CSVs

Write all metric columns back into the per-language CSVs in `data/indic/`.
All downstream notebooks (`indic/04` onwards) will find the metric columns
alongside the text columns.

In [ ]:
METRIC_COLS = [
    "comet", "comet_rom",
    "bleurt", "bleurt_rom",
    "bertscore_f1", "bertscore_f1_rom",
    "bertscore_p",  "bertscore_p_rom",
    "bertscore_r",  "bertscore_r_rom",
]

for iso, df in data.items():
    out_path = DATA_DIR / f"{iso}_indicmt.csv"
    df.to_csv(out_path, index=False)
    present = [c for c in METRIC_COLS if c in df.columns]
    print(f"  Saved  {out_path}  ({len(df):,} rows)  metric cols: {present}")

print(f"\n  {len(data)} files updated in {DATA_DIR.resolve()}")

## Step 8 — Scoring Summary

Print mean COMET, BLEURT, and BERTScore-F1 for each language under both
conditions. These figures correspond to Table 2 of the paper (COMET columns)
and Table 8 (full metric–human correlation table).

In [ ]:
print(f"{'Lang':6s}  {'Cond':10s}  {'COMET':>8s}  {'BLEURT':>8s}  {'BS-F1':>8s}")
print("-" * 55)

for iso, df in data.items():
    for cond, suffix in [("native", ""), ("romanised", "_rom")]:
        comet_col = f"comet{suffix}"
        blrt_col  = f"bleurt{suffix}"
        bs_col    = f"bertscore_f1{suffix}"

        comet_m = f"{df[comet_col].mean():.2f}"  if comet_col in df.columns else "--"
        blrt_m  = f"{df[blrt_col].mean():.4f}"   if blrt_col  in df.columns else "--"
        bs_m    = f"{df[bs_col].mean():.4f}"      if bs_col    in df.columns else "--"

        print(f"  {iso:4s}    {cond:10s}  {comet_m:>8s}  {blrt_m:>8s}  {bs_m:>8s}")

print("\n  data/indic/ is ready for indic/04 (TP/IP/SBI/IPI computation).")